In [ ]:
import os
from models.resnet import ResNetPatchClassifier
from models.mil_classifier import MILAttentionClassifier
from main import extract_patches, extract_features_per_wsi
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from datasets.mildataset import WSIMILTestDataset


class bcolors:
    HEADER = "\033[95m"
    OKBLUE = "\033[94m"
    DEBUG = "\033[96m"
    INFO = "\033[95m"  # pink
    WARNING = "\033[93m"  # yellow
    ERROR = "\033[91m"
    ENDC = "\033[0m"
    BOLD = "\033[1m"
    UNDERLINE = "\033[4m"


def test_mil_classifier(model, feature_dir, device):
    model.eval()
    model.to(device)

    test_dataset = WSIMILTestDataset(feature_dir)
    # Using batch_size=1 for WSI because each "bag" is one WSI
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    all_predictions = []
    all_true_labels = []
    wsi_names = []

    print(f"{bcolors.INFO}[INFO]{bcolors.ENDC} Starting model testing...")

    with torch.no_grad():
        for features, labels, wsi_name in test_loader:
            # Features are a list of tensors, each tensor is a bag of patches for one WSI
            # In our DataLoader, batch_size=1, so features will be a list containing one tensor
            features = features.squeeze(0).to(
                device
            )  # Remove batch dimension, move to device
            labels = labels.to(device)

            output = model(features)  # Output is logits for each class
            probabilities = (
                torch.softmax(output, dim=1)[:, 1].cpu().item()
            )  # Probability of positive class (tumor)
            predicted_label = (
                probabilities > 0.5
            ).long()  # Binary prediction based on 0.5 threshold

            all_predictions.append(probabilities)
            all_true_labels.append(labels.cpu().item())
            wsi_names.append(wsi_name[0])  # wsi_name is a tuple

            print(
                f"WSI: {wsi_name[0]}, True Label: {labels.cpu().item()}, Predicted Probability (Tumor): {probabilities:.4f}, Predicted Class: {predicted_label.item()}"
            )

    auc_score = roc_auc_score(all_true_labels, all_predictions)
    # Convert probabilities to binary predictions for other metrics
    binary_predictions = [1 if p > 0.5 else 0 for p in all_predictions]
    accuracy = accuracy_score(all_true_labels, binary_predictions)
    precision = precision_score(all_true_labels, binary_predictions, zero_division=0)
    recall = recall_score(all_true_labels, binary_predictions, zero_division=0)
    f1 = f1_score(all_true_labels, binary_predictions, zero_division=0)

    print(f"\n{bcolors.OKGREEN}--- Test Results ---{bcolors.ENDC}")
    print(f"Test AUC: {auc_score:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Precision: {precision:.4f}")
    print(f"Test Recall: {recall:.4f}")
    print(f"Test F1-Score: {f1:.4f}")

    results_df = pd.DataFrame(
        {
            "WSI_Name": wsi_names,
            "True_Label": all_true_labels,
            "Predicted_Probability": all_predictions,
            "Predicted_Class": binary_predictions,
        }
    )
    results_df.to_csv("mil_test_results.csv", index=False)
    print(
        f"{bcolors.INFO}[INFO]{bcolors.ENDC} Detailed results saved to mil_test_results.csv"
    )

    return {
        "auc": auc_score,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }


# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{bcolors.INFO}[INFO]{bcolors.ENDC} Using device: {device}")

# Step 1: Extract patches for test images
print(f"{bcolors.INFO}[INFO]{bcolors.ENDC} Starting test patch extraction...")
extract_patches(level=3, test=True)
print(f"{bcolors.OKGREEN}Test patch extraction complete.{bcolors.ENDC}")

patch_feature_extractor = ResNetPatchClassifier().to(device)

test_patch_root_dir = os.path.join(
    os.getcwd(), "data", "camelyon16", "patches", "test", "level_3"
)
test_feature_save_dir = os.path.join(
    os.getcwd(), "data", "camelyon16", "features", "test", "level_3"
)
extract_features_per_wsi(
    test=True, level=3, model_name="resnet18_patch_classifier_final_20250710055900"
)
print(f"{bcolors.OKGREEN}Test feature extraction complete.{bcolors.ENDC}")


# Step 3: Load the best MIL model
model_path = "src/models/mil_classifier_attention_best_level3_resnet18_patch_classifier_final_level3_20250710055900.pth.pth"
mil_model = MILAttentionClassifier(
    feature_dim=512, num_classes=2
)  # Feature dim for ResNet18
mil_model.load_state_dict(torch.load(model_path, map_location=device))
print(f"{bcolors.INFO}[INFO]{bcolors.ENDC} Loaded MIL model from {model_path}")

# Step 4: Run test evaluation
results = test_mil_classifier(mil_model, test_feature_save_dir, device)
print(f"Final Test AUC: {results['auc']:.4f}")